In [20]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=500, n_features=20, n_informative=8,
                            n_redundant=4, weights=[0.7, 0.3])
df = pd.DataFrame(X)
df["target"] = y

df.to_csv("/tmp/quick_data.csv", index=False)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3)

m = RandomForestClassifier(n_estimators=137, max_depth=11)
m.fit(Xtr, ytr)
print("score:", m.score(Xte, yte))

# ¯\_(ツ)_/¯

score: 0.8666666666666667


1. kein SEED -> nicht reproduzierbar
2. hardcoded Pfad (bricht auf Windows, .... )
3. keine stratify 
4. kein Tracking 
5. nur eine Metrik wird ausgerechnet 
6. alles in einer Zelle 
7. keine Begründung/nicht ersichtlich

In [21]:
import mlflow
import numpy as np 
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)


# Annahme dataset wurde_gehasht
ds_hash = "123456789_hash"

In [22]:
X, y = make_classification(n_samples=500, n_features=20, n_informative=8,
                            n_redundant=4, weights=[0.7, 0.3], random_state=RANDOM_SEED)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED)



In [23]:
EXPERIMENT_NAME = "lab1-rf"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run() as run:
    params = {"n_estimators": 200 , "max_depth": 12, "random_state": RANDOM_SEED }
    mlflow.log_params(params)
    mlflow.set_tag("dataset_hash", ds_hash)
    mlflow.set_tag("phase", "baseline")
    m = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=params["random_state"])

    m.fit(Xtr, ytr)
    y_pred = m.predict(Xte)
    y_pred_proba = m.predict_proba(Xte)[:,1]   

    metrics = {
        "accuracy": accuracy_score(yte, y_pred),
        "f1": f1_score(yte, y_pred),
        "roc_auc": roc_auc_score(yte, y_pred_proba)
    }

    mlflow.log_metrics(metrics)

    signature = mlflow.models.infer_signature(Xtr[:5], m.predict(Xtr[:5]))
    mlflow.sklearn.log_model(
        sk_model=m,
        artifact_path="model",
        signature=signature,
        input_example=Xtr[:5]
    )
    baseline_run_id = run.info.run_id
    print("Baseline run_id:", baseline_run_id)
    print("Metriken;", metrics)

Baseline run_id: 15fed6c578a442d88b2ea1b856cefbef
Metriken; {'accuracy': 0.8733333333333333, 'f1': 0.8041237113402062, 'roc_auc': 0.9058548920443493}


In [24]:
signature = mlflow.models.infer_signature(Xte[:5], m.predict(Xte[:5]))
signature

inputs: 
  [Tensor('float64', (-1, 20))]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None

In [ ]:
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True, silent=True)

with mlflow.start_run(run_name="autolog-baseline") as run:
    mlflow.set_tag("dataset_hash", ds_hash)
    mlflow.set_tag("phase", "autolog")
    rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_SEED)
    rf.fit(Xtr, ytr)
    autolog_run_id = run.info.run_id
    print("Autolog run_id:", autolog_run_id)

mlflow.sklearn.autolog(disable=True)  